# **Máster en Behavioral Data Science**
## **Instituto de Formación Continua (IL3) - Universitat de Barcelona**
## **Módulo 9: Aprendizaje Profundo - Reto 2** - (Notebook 2/4)
Autor: **Meysam Madadi**

Colaborador: **Julio C. S. Jacques Junior**

---

# **Los objetivos de este Jupyter notebook**
- Crear nuestro primer modelo utilizando información de texto como entrada.
- Comparar dos estrategias de fusión de características:
 - utilizando un modelo secuencial (LSTM)
 - utilizando el valor promedio
- Predecir la personalidad aparente a partir de datos textuales.
- Visualizar los resultados.

## **Descargando y descomprimiendo los datos**
- Aunque se descargue el conjunto de datos preprocesado completo, en este *notebook* solo trabajaremos con datos textuales (es decir, características de transcripción de video extraídas por BERT).

In [ ]:
# Download and unzip the data
!wget https://data.chalearnlap.cvc.uab.cat/Colab_MFPDS/2024BehaviorDSMaster/M9_r2/data_final.zip
!unzip ./data_final.zip

## Importando las librerías necesarias para ejecutar el código
- Los modelos están implementados y entrenados usando la librería Keras.

In [ ]:
import tensorflow as tf
from tensorflow.keras import optimizers
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout, BatchNormalization, ReLU, LeakyReLU, LSTM
from keras import regularizers
import numpy as np

## **Definiendo nuestra clase "DataGenerator" para cargar los datos en lotes**

- Esta es una forma estándar en Keras de cargar los datos por lotes en casos donde el conjunto de datos es demasiado grande para caber en la memoria. Por lo tanto, es necesario leer/cargar los datos secuencialmente desde el disco.

- La clase "DataGenerator" debe tener al menos dos funciones: **\_\_len__()** para devolver el número de lotes (*batches*) en cada "*epoch*" y **\_\_getitem(step)__** para devolver los datos del lote para un determinado paso de entranamiento, como "X" e "y", donde "X" es la lista de entradas e "y" son las etiquetas.

En nuestra implementación, "DataGenerator" se inicializa con diferentes variables de entrada:
- **data_list** es una lista de nombres de archivos de video para conjuntos de entrenamiento, validación o prueba,
- **root** es la ruta al conjunto de destino (es decir, entrenamiento, validación o prueba),
- **is_sequence** indica si las funciones de transcripción se dan como una secuencia de *tokens* o como promedio,
- **batch_size** es el tamaño del lote,
- **shuffle** se refiere a la aleatorización del orden de los ejemplos de entrenamiento antes de enviarlos a la función de entrenamiento durante cada *epoch*. Es habitual desordenar el orden de las muestras de entrenamento para que el algoritmo de aprendizaje reciba un orden diferente de muestras en cada *epoch*.

En este *Jupyter Notebook*, tratamos las características de transcripción de dos maneras:
- **Estrategia 1:** como una secuencia de "*tokens*". En este caso, los datos tienen una forma de **(batch_size, F, 768)** donde F es el número de "*tokens*" y 768 es la dimensionalidad de las características dadas por BERT. Sin embargo, dado que cada transcripción tiene un número diferente de "*tokens*", fijamos F para que sea el número máximo posible de "*tokens*" (=114). Para transcripciones con un número menor de "*tokens*" que 114, llenamos los que faltan con cero para asegurarnos de tener siempre un tensor de entrada de forma fija.
- **Estrategia 2:** los *tokens* se fusionan mediante una operación de promedio simple. En este caso, los datos tienen una forma de (batch_size, 768).

**Nota importante:** debemos tener en cuenta que podemos cargar nuestro conjunto de datos en un modelo de aprendizaje profundo sin la necesidad de implementar una función "DataGenerator" compleja. Esto dependerá de los recursos computacionales que tengamos (GPU) y del tamaño de nuestro conjunto de datos. El método del generador de datos utilizado en este reto se propuso como una alternativa para lidiar con las limitaciones de Colab. Podéis consultar la documentación de Keras para aprender diferentes (y más simples) formas de cargar los datos durante el entrenamiento, que podrían utilizarse en otros contextos.

In [ ]:
class DataGenerator(tf.keras.utils.Sequence):
    'Generates data for Keras'
    def __init__(self, data_list, root, is_sequence=False, batch_size=16, shuffle=True):
        super().__init__()
        'Initialization'
        self.data_list = data_list
        self.root = root
        self.is_sequence = is_sequence
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        'Denotes the number of batches per epoch'
        return int(np.floor(len(self.data_list) / self.batch_size))

    def __getitem__(self, index):
        'Generate one batch of data'
        # Generate indexes of the batch
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]

        # Find list of IDs
        data_list_temp = [self.data_list[k] for k in indexes]

        # Generate data
        X, Y = self.__data_generation(data_list_temp)

        return X, Y

    def on_epoch_end(self):
        'Updates indexes after each epoch'
        self.indexes = np.arange(len(self.data_list))
        if self.shuffle == True:
            np.random.shuffle(self.indexes)

    def __data_generation(self, data_list):
      'Generates data containing batch_size samples' # X : (batch_size, 114, 768) or (batch_size, 768)
      X, Y = [], []
      n_max = 114 # the possible maximum number of tokens in a cell
      for f in data_list:
        features = np.load(open(self.root+f+'/transcription_features.npy', 'rb'))[0]
        if self.is_sequence:
          n_seq, n_feat = features.shape
          # if the number of tokens is smaller than the maximum number of tokens,...
          # we pad the features with zero to have a fixed length sequence
          if n_seq < n_max:
            features = np.concatenate([np.zeros((n_max - n_seq, n_feat)), features], axis=0)
        else:
          # the features of all words are averaged
          features = np.mean(features, axis=0)
        X.append(features)

        # ['extraversion', 'neuroticism', 'agreeableness', 'conscientiousness', 'openness']
        traits = np.load(open(self.root+f+'/annotation.npy', 'rb'))
        Y.append(traits)

      return np.array(X, np.float32), np.array(Y, np.float32)

## **Construyendo nuestro modelo de red neuronal: solo información textual**

- En función de si queremos un modelo secuencial o no podemos diseñar la red. Para esto, podemos configurar la variable **"is_sequential"** como **True** o **False** en la siguiente celda.
- En el caso de características secuenciales (estrategia 1), definimos dos capas recurrentes en forma de LSTM ("*long short term memory*"). En la primera capa LSTM, las características actualizadas por *token* se devuelven con una forma de salida de (batch_size, 114, 768). En la segunda capa LSTM, solo se devuelven las características actualizadas del último "*token*". Finalmente, se agrega un MLP para predecir los cinco rasgos de personalidad.
- Si el modelo no es secuencial (estrategia 2), la red es un MLP (perceptrón multi-capas) simple con un solo vector promedio de *tokens* de entrada.
- Al final de la siguiente celda mostramos una representación visual de nuestro modelo.


In [ ]:
is_sequential = True

# Create model
def create_model(is_sequential):

    model = Sequential()

    if is_sequential:
      model.add(tf.keras.Input(shape=(114, 768)))

      model.add(LSTM(768,input_shape=(114, 768), activation=None, return_sequences=True))
      model.add(Dropout(0.2))

      model.add(LSTM(768, activation=None))
      model.add(Dropout(0.2))
    else:
      model.add(tf.keras.Input(shape=(768,)))

    # MLP
    model.add(Dense(1024, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.2))

    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())

    model.add(Dense(5, activation='sigmoid'))

    return model

model = create_model(is_sequential)
#print(model.summary())
tf.keras.utils.plot_model(model, show_shapes=True)

## **Leyendo la lista de archivos de entrenamiento, validación y prueba**

In [ ]:
# Read the data lists

with open('train.txt', 'r') as f:
  train_list = f.readlines()
  for i in range(len(train_list)):
    train_list[i] = train_list[i].rsplit('\n',1)[0]

with open('validation.txt', 'r') as f:
  validation_list = f.readlines()
  for i in range(len(validation_list)):
    validation_list[i] = validation_list[i].rsplit('\n',1)[0]

with open('test.txt', 'r') as f:
  test_list = f.readlines()
  for i in range(len(test_list)):
    test_list[i] = test_list[i].rsplit('\n',1)[0]

## **Entrenando y evaluando nuestro modelo**

El modelo se entrena con el optimizador Adam, con una tasa de aprendizaje de 1e-5 y una función de pérdida de error cuadrático medio (L2). El tamaño del lote es 32, y el modelo se entrena durante 20 *epochs*. Se define un "callback" para guardar el mejor modelo entrenado en función del error absoluto medio observado en el conjunto de validación. Finalmente, se guarda el registro del historial en la ruta definida y se evalúa el modelo en el conjunto de pruebas.


In [ ]:
# Training
import gc
import random
import pickle

lr = 1e-5
batch_size = 32
n_epochs = 20
checkpoint = './best_model_text_sequential.h5' if is_sequential else './best_model_text_avg.h5'
shuffle = True
verbose = 1

# creating data generators to load the data
train_dg = DataGenerator(train_list, './data_final/train/', is_sequence=is_sequential, batch_size=batch_size, shuffle=shuffle)
validation_dg = DataGenerator(validation_list, './data_final/validation/', is_sequence=is_sequential, batch_size=batch_size, shuffle=False)
test_dg = DataGenerator(test_list, './data_final/test/', is_sequence=is_sequential, batch_size=batch_size, shuffle=False)

# defining the optimizer
model.compile(tf.keras.optimizers.Adam(learning_rate=lr), loss=tf.keras.losses.MeanSquaredError(), metrics=['mae'])

# saving the best model based on val_loss
mc = tf.keras.callbacks.ModelCheckpoint(checkpoint, monitor='val_mae', mode='min', save_best_only=True)

# training the model and saving the history
history = model.fit(train_dg, validation_data=validation_dg, epochs=n_epochs, verbose=verbose, callbacks=[mc])
with open('./train_history_text_sequential.pkl' if is_sequential else './train_history_text_avg.pkl', 'wb') as handle:
  pickle.dump(history.history, handle, protocol=pickle.HIGHEST_PROTOCOL)


In [ ]:
# Ceating/building our model again, and loading the last checkpoint (best model)
model = create_model(is_sequential)
model.load_weights(checkpoint)
model.compile(tf.keras.optimizers.Adam(learning_rate=lr), loss=tf.keras.losses.MeanSquaredError(), metrics=['mae'])

# Evaluate the trained model on the test set

# Some house keeping
gc.collect()
tf.keras.backend.clear_session()

print('Evaluating on the test set')

_loss = 0
_mae = 0
for step in range(test_dg.__len__()):
  # Load the batch
  X, Y = test_dg.__getitem__(step)

  # validate on one batch
  loss, mae = model.evaluate(
      tf.convert_to_tensor(X, dtype=tf.float32),
      tf.convert_to_tensor(Y, dtype=tf.float32),
      verbose = 0)

  _loss += loss
  _mae += mae
step += 1
print("The final mean absolute error is {0:.5f}\n".format(_mae/step))

## **Visualizando las curvas de entrenamiento**
- Pérdida de entrenamiento y validación, y MAE para cada *epoch*.
- Las curvas de entrenamiento pueden ayudarnos a entender un poco el comportamiento del modelo. Por ejemplo, si el modelo se está sobreajustando, o si el modelo aún estaba aprendiendo al final del entrenamiento, etc.

In [ ]:
# Visualization
from matplotlib import pyplot as plt

# Load the history
train_hist = pickle.load(open('./train_history_text_sequential.pkl' if is_sequential else './train_history_text_avg.pkl',"rb"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))
fig.suptitle('Training history', fontsize=14, fontweight='bold')

ax1.plot(train_hist['loss'])
ax1.plot(train_hist['val_loss'])
ax1.set(xlabel='epoch', ylabel='Loss')
ax1.legend(['train', 'valid'], loc='upper right')

ax2.plot(train_hist['mae'])
ax2.plot(train_hist['val_mae'])
ax2.set(xlabel='epoch', ylabel='MAE')
ax2.legend(['train', 'valid'], loc='upper right')